# Week 2, Day 1: One client, three providers

The same Chat Completions call against Gemini, a local Ollama server, and OpenRouter.

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(override=True)

gemini_api_key = os.getenv("GEMINI_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

for name, key in [("GEMINI_API_KEY", gemini_api_key), ("OPENROUTER_API_KEY", openrouter_api_key)]:
    print(f"{name} found" if key else f"{name} not set")

gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=gemini_api_key)
openrouter = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

GEMINI = "gemini-3.1-flash-lite"
LLAMA = "llama3.2"
QWEN = "qwen2.5:3b"
NEMOTRON = "nvidia/nemotron-3-ultra-550b-a55b:free"
LAGUNA = "poolside/laguna-s-2.1:free"
GLM = "z-ai/glm-4.5"


def ask(client, model, messages):
    response = client.chat.completions.create(model=model, messages=messages)
    display(Markdown(f"**{model}**\n\n{response.choices[0].message.content}"))
    return response

## The same prompt on each provider

In [ ]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

for client, model in [(gemini, GEMINI), (ollama, LLAMA), (openrouter, NEMOTRON)]:
    ask(client, model, tell_a_joke)

## Inference-time reasoning

`reasoning_effort` asks Gemini to spend more compute on the answer. The question is a classic:
two coins are tossed and one of them is heads. The probability the other is tails is 2/3, not 1/2.

In [ ]:
easy_puzzle = [
    {"role": "user", "content": "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

for model, effort in [("gemini-3.5-flash", "minimal"), ("gemini-3.5-flash", "low"), (GEMINI, "minimal")]:
    response = gemini.chat.completions.create(model=model, messages=easy_puzzle, reasoning_effort=effort)
    display(Markdown(f"**{model} / {effort}**\n\n{response.choices[0].message.content}"))

## A harder puzzle, across Gemini, Ollama, and OpenRouter

In [ ]:
hard_puzzle = [
    {"role": "user", "content": '''
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
'''},
]

for client, model in [(gemini, GEMINI), (ollama, LLAMA), (ollama, QWEN), (openrouter, NEMOTRON), (openrouter, LAGUNA)]:
    ask(client, model, hard_puzzle)

## A one-shot prisoner's dilemma

In [ ]:
dilemma = [
    {"role": "user", "content": '''
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
'''},
]

for client, model in [(gemini, GEMINI), (ollama, LLAMA), (ollama, QWEN), (openrouter, NEMOTRON)]:
    ask(client, model, dilemma)

## Ollama locally

The next cell should print `Ollama is running`. If it doesn't, start `ollama serve` in a terminal.

In [ ]:
import requests

requests.get("http://localhost:11434/").content

In [ ]:
!ollama pull llama3.2
!ollama pull qwen2.5:3b

In [ ]:
for model in [LLAMA, QWEN]:
    ask(ollama, model, easy_puzzle)

## OpenRouter as a router to another model

In [ ]:
ask(openrouter, GLM, tell_a_joke)

## LiteLLM: token usage on Gemini

LiteLLM is a thin wrapper that also reports cost. The model string is `gemini/` plus the model name.

In [ ]:
from litellm import completion

response = completion(model=f"gemini/{GEMINI}", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

usage = response.usage
cost = response._hidden_params.get("response_cost") or 0
print(f"Input tokens: {usage.prompt_tokens}")
print(f"Output tokens: {usage.completion_tokens}")
print(f"Total tokens: {usage.total_tokens}")
print(f"Total cost: {cost * 100:.4f} cents")

## Prompt caching with Gemini

Ask the question once, then again with the full text of Hamlet attached, then a third time.
The third call should report cached input tokens.

In [ ]:
with open("hamlet.txt", "r", encoding="utf-8") as f:
    hamlet = f.read()

loc = hamlet.find("Speak, man")
print(hamlet[loc:loc + 100])

question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]


def show_usage(response):
    usage = response.usage
    cached = getattr(usage.prompt_tokens_details, "cached_tokens", None)
    cost = response._hidden_params.get("response_cost") or 0
    print(f"Input tokens: {usage.prompt_tokens}")
    print(f"Output tokens: {usage.completion_tokens}")
    print(f"Cached tokens: {cached}")
    print(f"Total cost: {cost * 100:.4f} cents")

In [ ]:
response = completion(model=f"gemini/{GEMINI}", messages=question)
display(Markdown(response.choices[0].message.content))
show_usage(response)

In [ ]:
question[0]["content"] += "\n\nFor context, here is the entire text of Hamlet:\n\n" + hamlet

response = completion(model=f"gemini/{GEMINI}", messages=question)
display(Markdown(response.choices[0].message.content))
show_usage(response)

In [ ]:
response = completion(model=f"gemini/{GEMINI}", messages=question)
display(Markdown(response.choices[0].message.content))
show_usage(response)

## Adversarial chat

Gemini argues. Nemotron on OpenRouter, and Llama locally, try to stay polite.

In [ ]:
gemini_system = (
    "You are a chatbot who is very argumentative; "
    "you disagree with anything in the conversation and you challenge everything, in a snarky way."
)
polite_system = (
    "You are a very polite, courteous chatbot. You try to agree with "
    "everything the other person says, or find common ground. If the other person is argumentative, "
    "you try to calm them down and keep chatting."
)


def call_gemini(other_messages):
    messages = [{"role": "system", "content": gemini_system}]
    for mine, theirs in zip(gemini_messages, other_messages):
        messages.append({"role": "assistant", "content": mine})
        messages.append({"role": "user", "content": theirs})
    response = gemini.chat.completions.create(model=GEMINI, messages=messages)
    return response.choices[0].message.content


def call_partner(client, model, partner_messages):
    messages = [{"role": "system", "content": polite_system}]
    for mine, theirs in zip(gemini_messages, partner_messages):
        messages.append({"role": "user", "content": mine})
        messages.append({"role": "assistant", "content": theirs})
    messages.append({"role": "user", "content": gemini_messages[-1]})
    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content


def converse(client, model, partner_name, rounds=5):
    global gemini_messages
    gemini_messages = ["Hi there"]
    partner_messages = ["Hi"]
    display(Markdown(f"### Gemini\n{gemini_messages[0]}"))
    display(Markdown(f"### {partner_name}\n{partner_messages[0]}"))
    for _ in range(rounds):
        gemini_next = call_gemini(partner_messages)
        gemini_messages.append(gemini_next)
        display(Markdown(f"### Gemini\n{gemini_next}"))
        partner_next = call_partner(client, model, partner_messages)
        partner_messages.append(partner_next)
        display(Markdown(f"### {partner_name}\n{partner_next}"))

In [ ]:
converse(openrouter, NEMOTRON, "Nemotron")

In [ ]:
converse(ollama, LLAMA, "Llama")

## Three kids talking about their day

Mia is Gemini, Leo is Llama 3.2, and Sam is Qwen 2.5 3B.

In [ ]:
kids = [
    {"name": "Mia", "model": GEMINI, "client": gemini, "friends": "Leo and Sam"},
    {"name": "Leo", "model": LLAMA, "client": ollama, "friends": "Mia and Sam"},
    {"name": "Sam", "model": QWEN, "client": ollama, "friends": "Mia and Leo"},
]

conversation = "Mia: Hi! What did you guys do today?"
display(Markdown("### Mia\nHi! What did you guys do today?"))


def call_kid(kid):
    messages = [
        {"role": "system", "content": (
            f"You are {kid['name']}, a kid chatting with your friends {kid['friends']}. "
            "You talk like a real kid about an ordinary day: school, recess, lunch, games, and what happened at home. "
            "Keep each reply to one or two short sentences. Reply with only what you say next, with no name prefix."
        )},
        {"role": "user", "content": (
            f"You are {kid['name']}, in conversation with {kid['friends']}.\n"
            f"The conversation so far is as follows:\n{conversation}\n"
            f"Now respond with what you would like to say next, as {kid['name']}."
        )},
    ]
    response = kid["client"].chat.completions.create(model=kid["model"], messages=messages)
    return response.choices[0].message.content


for _ in range(3):
    for kid in kids:
        reply = call_kid(kid)
        conversation += f"\n{kid['name']}: {reply}"
        display(Markdown(f"### {kid['name']}\n{reply}"))